# MotoGP Riders Finishing Positions - Data Preparation

This notebook cleans and standardizes the finishing positions dataset based on insights from the exploration phase.

## 0. Setup and Data Loading

In [ ]:
# Import necessary libraries
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configure plot styles
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Load raw data
raw_data_path = Path("../data/raw/riders_finishing_positions.csv")
df_raw = pd.read_csv(raw_data_path)

print(f"Raw data loaded: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head()

## 1. Data Quality Assessment

In [ ]:
# Assess data quality
print("=== DATA QUALITY ASSESSMENT ===")
print(f"Total riders: {len(df_raw)}")
print(f"Unique riders: {df_raw['Rider'].nunique()}")
print(f"Unique countries: {df_raw['Country'].nunique()}")

print(f"\nMissing values:")
print(df_raw.isnull().sum())

print(f"\nData types:")
print(df_raw.dtypes)

print(f"\nBasic statistics:")
numeric_columns = ['Victories', 'NumberofSecond', 'NumberofThird', 'Numberof4th', 'Numberof5th', 'Numberof6th']
print(df_raw[numeric_columns].describe())

## 2. Column Standardization

In [ ]:
# Create working copy and standardize columns
df = df_raw.copy()

# Standardize column names
column_mapping = {
    'Rider': 'rider',
    'Victories': 'victories',
    'NumberofSecond': 'second_places',
    'NumberofThird': 'third_places',
    'Numberof4th': 'fourth_places',
    'Numberof5th': 'fifth_places',
    'Numberof6th': 'sixth_places',
    'Country': 'country'
}

df = df.rename(columns=column_mapping)

print("Standardized columns:")
print(list(df.columns))
df.head()

## 3. Data Type Corrections and Missing Value Treatment

In [ ]:
# Ensure proper data types for numerical columns
position_columns = ['victories', 'second_places', 'third_places', 'fourth_places', 'fifth_places', 'sixth_places']

for col in position_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Clean text fields
df['rider'] = df['rider'].astype(str).str.strip()
df['country'] = df['country'].astype(str).str.strip().str.upper()

# Handle missing values in position columns (fill with 0)
print(f"Missing values before treatment:")
print(df[position_columns].isnull().sum())

for col in position_columns:
    df[col] = df[col].fillna(0).astype(int)

print(f"\nMissing values after treatment:")
print(df[position_columns].isnull().sum())

print(f"\nUpdated data types:")
print(df.dtypes)

## 4. Rider Name Standardization

In [ ]:
# Standardize rider names
print("=== RIDER NAME STANDARDIZATION ===")
print(f"Original unique riders: {df['rider'].nunique()}")

# Clean rider names
df['rider_clean'] = df['rider'].str.strip()

# Extract first and last names
df['rider_first_name'] = df['rider_clean'].str.split().str[0]
df['rider_last_name'] = df['rider_clean'].str.split().str[-1]
df['rider_display_name'] = df['rider_last_name'] + ', ' + df['rider_first_name']

# Show top riders by total finishes
df['total_finishes'] = df[position_columns].sum(axis=1)
top_riders = df.nlargest(20, 'total_finishes')[['rider_clean', 'country', 'victories', 'total_finishes']]
print(f"\nTop 20 riders by total race finishes:")
print(top_riders.to_string(index=False))

# Check for potential duplicates
duplicate_riders = df['rider_clean'].duplicated()
if duplicate_riders.any():
    print(f"\nWarning: {duplicate_riders.sum()} potential duplicate rider names found")
    print(df[duplicate_riders][['rider_clean', 'country']].values)
else:
    print("\nNo duplicate rider names found")

print(f"Final unique riders: {df['rider_clean'].nunique()}")

## 5. Country Standardization and Continental Mapping

In [ ]:
# Standardize countries and add continental mapping
print("=== COUNTRY STANDARDIZATION ===")

df['country_clean'] = df['country'].str.strip().str.upper()

# Show country distribution
print(f"Country distribution by riders:")
country_rider_counts = df['country_clean'].value_counts()
print(country_rider_counts.head(15))

# Continental mapping (same as used in other datasets)
continent_mapping = {
    # Europe
    'IT': 'Europe', 'ES': 'Europe', 'GB': 'Europe', 'FR': 'Europe', 'DE': 'Europe', 
    'NL': 'Europe', 'CH': 'Europe', 'AT': 'Europe', 'CZ': 'Europe', 'FI': 'Europe',
    'SE': 'Europe', 'NO': 'Europe', 'DK': 'Europe', 'BE': 'Europe', 'IE': 'Europe',
    'PT': 'Europe', 'GR': 'Europe', 'HU': 'Europe', 'PL': 'Europe', 'SI': 'Europe',
    
    # Asia
    'JP': 'Asia', 'MY': 'Asia', 'TH': 'Asia', 'CN': 'Asia', 'IN': 'Asia', 'ID': 'Asia',
    'KR': 'Asia', 'TW': 'Asia', 'PH': 'Asia', 'SG': 'Asia', 'VN': 'Asia', 'QA': 'Asia',
    
    # Oceania
    'AU': 'Oceania', 'NZ': 'Oceania',
    
    # North America
    'US': 'North America', 'CA': 'North America', 'MX': 'North America',
    
    # South America
    'BR': 'South America', 'AR': 'South America', 'VE': 'South America',
    'CO': 'South America', 'CL': 'South America',
    
    # Africa
    'ZA': 'Africa', 'EG': 'Africa', 'MA': 'Africa'
}

df['continent'] = df['country_clean'].map(continent_mapping)
df['continent'] = df['continent'].fillna('Other')

print(f"\nContinent distribution:")
continent_counts = df['continent'].value_counts()
print(continent_counts)

# Check for unmapped countries
unmapped = df[df['continent'] == 'Other']['country_clean'].unique()
if len(unmapped) > 0:
    print(f"\nUnmapped countries: {unmapped}")
else:
    print("\nAll countries successfully mapped to continents")

## 6. Performance Metrics Creation

In [ ]:
# Create comprehensive performance metrics
print("=== PERFORMANCE METRICS CREATION ===")

# Basic aggregated metrics
df['total_races'] = df[position_columns].sum(axis=1)
df['total_podiums'] = df['victories'] + df['second_places'] + df['third_places']
df['points_finishes'] = df[position_columns].sum(axis=1)  # Assuming top 6 get points
df['non_podium_points'] = df['fourth_places'] + df['fifth_places'] + df['sixth_places']

# Performance rates (avoiding division by zero)
df['win_rate'] = np.where(df['total_races'] > 0, df['victories'] / df['total_races'], 0)
df['podium_rate'] = np.where(df['total_races'] > 0, df['total_podiums'] / df['total_races'], 0)
df['points_rate'] = np.where(df['total_races'] > 0, df['points_finishes'] / df['total_races'], 1)  # Assuming all recorded are points

# Podium efficiency (victories as % of total podiums)
df['podium_win_rate'] = np.where(df['total_podiums'] > 0, df['victories'] / df['total_podiums'], 0)

# Performance categories
def categorize_rider_performance(row):
    if row['victories'] >= 20:
        return 'Elite Winner'
    elif row['victories'] >= 5:
        return 'Regular Winner'
    elif row['victories'] >= 1:
        return 'Race Winner'
    elif row['total_podiums'] >= 5:
        return 'Podium Finisher'
    elif row['total_podiums'] >= 1:
        return 'Occasional Podium'
    elif row['non_podium_points'] >= 10:
        return 'Points Scorer'
    else:
        return 'Limited Results'

df['performance_category'] = df.apply(categorize_rider_performance, axis=1)

print("Performance metrics created:")
new_metrics = ['total_races', 'total_podiums', 'points_finishes', 'non_podium_points',
               'win_rate', 'podium_rate', 'points_rate', 'podium_win_rate', 'performance_category']
print(new_metrics)

print(f"\nPerformance category distribution:")
print(df['performance_category'].value_counts())

## 7. Data Validation and Consistency Checks

In [ ]:
# Comprehensive data validation
print("=== DATA VALIDATION ===")

# Check that sum relationships are correct
calculated_total = df[position_columns].sum(axis=1)
mismatch = (calculated_total != df['total_races']).sum()
if mismatch > 0:
    print(f"Warning: {mismatch} riders with inconsistent total race calculations")
else:
    print("✓ Total races calculations are consistent")

# Check that podium calculations are correct
calculated_podiums = df['victories'] + df['second_places'] + df['third_places']
podium_mismatch = (calculated_podiums != df['total_podiums']).sum()
if podium_mismatch > 0:
    print(f"Warning: {podium_mismatch} riders with inconsistent podium calculations")
else:
    print("✓ Podium calculations are consistent")

# Check for negative values
for col in position_columns:
    negative_count = (df[col] < 0).sum()
    if negative_count > 0:
        print(f"Warning: {negative_count} negative values in {col}")
    else:
        print(f"✓ No negative values in {col}")

# Check rate calculations (should be between 0 and 1)
rate_columns = ['win_rate', 'podium_rate', 'points_rate', 'podium_win_rate']
for col in rate_columns:
    invalid_rates = ((df[col] < 0) | (df[col] > 1)).sum()
    if invalid_rates > 0:
        print(f"Warning: {invalid_rates} invalid rates in {col}")
        print(f"Range: {df[col].min():.3f} - {df[col].max():.3f}")
    else:
        print(f"✓ All {col} values are valid (0-1)")

print(f"\n=== VALIDATION SUMMARY ===")
print(f"Total riders after preparation: {len(df)}")
print(f"Riders with at least one race finish: {(df['total_races'] > 0).sum()}")
print(f"Riders with at least one victory: {(df['victories'] > 0).sum()}")
print(f"Riders with at least one podium: {(df['total_podiums'] > 0).sum()}")
print(f"Countries represented: {df['country_clean'].nunique()}")

## 8. Cross-Reference with Riders Info Dataset

In [ ]:
# Cross-reference with riders_info dataset if available
print("=== CROSS-REFERENCE WITH RIDERS INFO ===")

try:
    # Try to load prepared riders_info dataset
    riders_info_path = Path("../../data/interim/riders_info_prepared.csv")
    if riders_info_path.exists():
        riders_info = pd.read_csv(riders_info_path)
        
        # Compare victory counts where possible
        # Note: These datasets may have different scopes, so differences are expected
        common_riders = []
        for rider in df['rider_clean']:
            # Simple name matching - could be improved with fuzzy matching
            matches = riders_info[riders_info['rider_name_clean'].str.contains(rider.split()[-1], case=False, na=False)]
            if len(matches) > 0:
                common_riders.append((rider, matches.iloc[0]['rider_name_clean'], 
                                    df[df['rider_clean'] == rider]['victories'].iloc[0],
                                    matches.iloc[0]['victories']))
        
        print(f"Found {len(common_riders)} potential matches between datasets")
        if len(common_riders) > 0:
            print("Sample comparisons (Finishing Positions vs Riders Info):")
            for i, (fp_name, ri_name, fp_victories, ri_victories) in enumerate(common_riders[:5]):
                print(f"{i+1}. {fp_name} vs {ri_name}: {fp_victories} vs {ri_victories} victories")
        
        print("\nNote: Differences are expected due to different dataset scopes and time periods")
    else:
        print("Riders info prepared dataset not found - skipping cross-reference")
        
except Exception as e:
    print(f"Error during cross-reference: {e}")
    print("Continuing without cross-reference")

## 9. Final Dataset Structure

In [ ]:
# Define final column order
final_columns = [
    # Rider identification
    'rider', 'rider_clean', 'rider_first_name', 'rider_last_name', 'rider_display_name',
    'country', 'country_clean', 'continent',
    # Position counts
    'victories', 'second_places', 'third_places', 'fourth_places', 'fifth_places', 'sixth_places',
    # Aggregated metrics
    'total_races', 'total_podiums', 'points_finishes', 'non_podium_points',
    # Performance rates
    'win_rate', 'podium_rate', 'points_rate', 'podium_win_rate',
    # Categories
    'performance_category'
]

# Create final dataset
df_final = df[final_columns].copy()

print("=== FINAL PREPARED DATASET ===")
print(f"Shape: {df_final.shape}")
print(f"Columns: {list(df_final.columns)}")

print(f"\nSample of prepared data:")
print(df_final.head(10))

print(f"\nData types:")
print(df_final.dtypes)

print(f"\nMissing values:")
print(df_final.isnull().sum())

## 10. Save Prepared Data

In [ ]:
# Create prepared data directory if it doesn't exist
prepared_data_path = Path("../../data/interim/")
prepared_data_path.mkdir(parents=True, exist_ok=True)

# Save prepared dataset
output_file = prepared_data_path / "finishing_positions_prepared.csv"
df_final.to_csv(output_file, index=False)

print(f"Prepared dataset saved to: {output_file}")
print(f"File size: {output_file.stat().st_size:,} bytes")

# Verification
df_verification = pd.read_csv(output_file)
print(f"\nVerification - loaded shape: {df_verification.shape}")
print("✓ File saved and verified successfully")

## 11. Preparation Summary

In [ ]:
print("=== PREPARATION SUMMARY ===")
print("\n✅ COMPLETED TASKS:")
print("1. Column name standardization")
print("2. Data type corrections and missing value treatment")
print("3. Rider name cleaning and standardization")
print("4. Country standardization and continental mapping")
print("5. Comprehensive performance metrics creation")
print("6. Performance categorization")
print("7. Data validation and consistency checks")
print("8. Cross-reference validation with riders info")
print("9. Prepared dataset saved")

print("\n📊 DATASET IMPROVEMENTS:")
print(f"- Added {len([col for col in df_final.columns if col not in df_raw.columns])} derived metrics")
print(f"- Standardized rider and country names")
print(f"- Created comprehensive performance rates")
print(f"- Added performance categories for analysis")
print(f"- Continental mapping for geographical analysis")

print("\n🏆 PERFORMANCE DISTRIBUTION:")
perf_dist = df_final['performance_category'].value_counts()
total_riders = len(df_final)
for category, count in perf_dist.items():
    percentage = (count / total_riders) * 100
    print(f"- {category}: {count} riders ({percentage:.1f}%)")

print("\n🌍 GEOGRAPHICAL DISTRIBUTION:")
geo_dist = df_final.groupby('continent').agg({
    'rider_clean': 'count',
    'victories': 'sum'
})
for continent, row in geo_dist.iterrows():
    riders = row['rider_clean']
    victories = row['victories']
    print(f"- {continent}: {riders} riders with {victories} total victories")

print("\n🔢 KEY STATISTICS:")
top_winner = df_final.nlargest(1, 'victories').iloc[0]
print(f"- Most victories: {top_winner['rider_clean']} ({top_winner['country_clean']}) with {top_winner['victories']} wins")
top_podiums = df_final.nlargest(1, 'total_podiums').iloc[0]
print(f"- Most podiums: {top_podiums['rider_clean']} with {top_podiums['total_podiums']} podiums")
best_win_rate = df_final[df_final['total_races'] >= 10].nlargest(1, 'win_rate').iloc[0]
print(f"- Best win rate (min 10 races): {best_win_rate['rider_clean']} with {best_win_rate['win_rate']:.3f}")

print("\n➡️  READY FOR ANALYSIS PHASE")
print("The prepared finishing positions dataset is now ready for rider performance analysis.")